# Phase 1 Review Slice

This notebook preserves the Phase 1 proof path with the current Phase 2 service signature.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

SEARCH_ROOTS = [
    Path.cwd().resolve(),
    Path('~/projects/onto-canon6').expanduser(),
]

PROJECT_ROOT = None
for start in SEARCH_ROOTS:
    candidate = start
    while True:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    if PROJECT_ROOT is not None:
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate onto-canon6 repo root from notebook cwd or ~/projects/onto-canon6"
    )

for candidate_path in (
    PROJECT_ROOT / "src",
    PROJECT_ROOT.parent / "llm_client",
    Path('~/projects/llm_client').expanduser(),
):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

from onto_canon6.pipeline import ReviewService  # noqa: E402


In [2]:
tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(tmp_dir.name) / "review.sqlite3"
service = ReviewService(db_path=db_path, default_acceptance_policy="record_only")
db_path

PosixPath('/tmp/tmpd6jxx21t/review.sqlite3')

In [3]:
valid_result = service.submit_candidate_assertion(
    payload={
        "predicate": "dodaf:activity_performs_resource",
        "roles": {
            "performer": [{"entity_id": "ent:performer:1", "entity_type": "dm2:Performer"}],
            "activity": [{"entity_id": "ent:activity:1", "entity_type": "dm2:OperationalActivity"}],
            "resource": [{"entity_id": "ent:resource:1", "entity_type": "dm2:Resource"}],
        },
    },
    profile_id="dodaf",
    profile_version="0.1.0",
    submitted_by="notebook:analyst",
    source_kind="notebook",
    source_ref="notebook://phase1/valid-dodaf",
    source_label="phase1 valid submission",
)
valid_result.model_dump()

{'candidate': {'candidate_id': 'cand_a7b479ea863d4fcf849a30d8',
  'profile': {'profile_id': 'dodaf', 'profile_version': '0.1.0'},
  'validation_status': 'valid',
  'review_status': 'pending_review',
  'payload_hash': 'sha256:f9dd398c9168dd46a47d08536a46135426c6838557ad108596fb81368526fd2b',
  'payload': {'predicate': 'dodaf:activity_performs_resource',
   'roles': {'activity': [{'entity_id': 'ent:activity:1',
      'entity_type': 'dm2:OperationalActivity',
      'kind': 'entity'}],
    'performer': [{'entity_id': 'ent:performer:1',
      'entity_type': 'dm2:Performer',
      'kind': 'entity'}],
    'resource': [{'entity_id': 'ent:resource:1',
      'entity_type': 'dm2:Resource',
      'kind': 'entity'}]}},
  'normalized_payload': {'predicate': 'dodaf:activity_performs_resource',
   'roles': {'activity': [{'entity_id': 'ent:activity:1',
      'entity_type': 'dm2:OperationalActivity',
      'kind': 'entity'}],
    'performer': [{'entity_id': 'ent:performer:1',
      'entity_type': 'dm2:P

In [4]:
mixed_result = service.submit_candidate_assertion(
    payload={
        "predicate": "oc:unknown_predicate_demo",
        "roles": {
            "subject": [{"entity_id": "ent:subject:1", "entity_type": "oc:person"}],
        },
    },
    profile_id="psyop_seed",
    profile_version="0.1.0",
    submitted_by="notebook:analyst",
    source_kind="notebook",
    source_ref="notebook://phase1/mixed-predicate",
    source_label="phase1 mixed submission",
)
mixed_result.model_dump()

{'candidate': {'candidate_id': 'cand_6a98976fcdce4db2997c74e5',
  'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
  'validation_status': 'needs_review',
  'review_status': 'pending_review',
  'payload_hash': 'sha256:a79b23766b1263e1d4b43058a110ab2731238253593a0835c3895d1610aee0fa',
  'payload': {'predicate': 'oc:unknown_predicate_demo',
   'roles': {'subject': [{'entity_id': 'ent:subject:1',
      'entity_type': 'oc:person',
      'kind': 'entity'}]}},
  'normalized_payload': {'predicate': 'oc:unknown_predicate_demo',
   'roles': {'subject': [{'entity_id': 'ent:subject:1',
      'entity_type': 'oc:person',
      'kind': 'entity'}]}},
  'validation': {'hard_errors': (),
   'soft_violations': ({'code': 'oc:profile_unknown_predicate',
     'message': "predicate 'oc:unknown_predicate_demo' is not declared in profile psyop_seed; mixed ontology mode routes it as a proposal",
     'path': 'predicate',
     'severity': 'soft'},),
   'type_checks_total': 0,
   'type_checks_

In [5]:
{
    "candidates": [candidate.model_dump() for candidate in service.list_candidate_assertions()],
    "pending_proposals": [proposal.model_dump() for proposal in service.list_proposals(status_filter="pending")],
}

{'candidates': [{'candidate_id': 'cand_a7b479ea863d4fcf849a30d8',
   'profile': {'profile_id': 'dodaf', 'profile_version': '0.1.0'},
   'validation_status': 'valid',
   'review_status': 'pending_review',
   'payload_hash': 'sha256:f9dd398c9168dd46a47d08536a46135426c6838557ad108596fb81368526fd2b',
   'payload': {'predicate': 'dodaf:activity_performs_resource',
    'roles': {'activity': [{'entity_id': 'ent:activity:1',
       'entity_type': 'dm2:OperationalActivity',
       'kind': 'entity'}],
     'performer': [{'entity_id': 'ent:performer:1',
       'entity_type': 'dm2:Performer',
       'kind': 'entity'}],
     'resource': [{'entity_id': 'ent:resource:1',
       'entity_type': 'dm2:Resource',
       'kind': 'entity'}]}},
   'normalized_payload': {'predicate': 'dodaf:activity_performs_resource',
    'roles': {'activity': [{'entity_id': 'ent:activity:1',
       'entity_type': 'dm2:OperationalActivity',
       'kind': 'entity'}],
     'performer': [{'entity_id': 'ent:performer:1',
      

In [6]:
accepted = service.review_proposal(
    proposal_id=mixed_result.proposals[0].proposal_id,
    decision="accepted",
    actor_id="notebook:reviewer",
    note_text="Reasonable local predicate for later ontology review.",
)
accepted.model_dump()

{'proposal_id': 'prop_19bfbdb65011c7f88b0e0ef9',
 'proposal_key': 'sha256:19bfbdb65011c7f88b0e0ef95d491dc8462f70bc461fec8f47d45468af7a1340',
 'proposal_kind': 'predicate',
 'proposed_value': 'oc:unknown_predicate_demo',
 'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
 'target_pack': {'pack_id': 'onto_canon_psyop_seed__overlay',
  'pack_version': '0.1.0'},
 'reason': "default ontology mode 'mixed' resolved action 'propose' for predicate",
 'status': 'accepted',
 'application_status': 'recorded',
 'details': {'candidate_source_kind': 'notebook',
  'source': 'validate_assertion_payload'},
 'candidate_ids': ('cand_6a98976fcdce4db2997c74e5',),
 'created_at': '2026-03-17T21:27:55.060120Z',
 'review': {'decision_id': 'dec_2a6dd71769814871bf077d19',
  'proposal_id': 'prop_19bfbdb65011c7f88b0e0ef9',
  'decision': 'accepted',
  'actor_id': 'notebook:reviewer',
  'note_text': 'Reasonable local predicate for later ontology review.',
  'acceptance_policy': 'record_only',
  'ap

In [7]:
{
    "accepted_proposals": [proposal.model_dump() for proposal in service.list_proposals(status_filter="accepted")],
    "all_candidates": [candidate.model_dump() for candidate in service.list_candidate_assertions()],
}

{'accepted_proposals': [{'proposal_id': 'prop_19bfbdb65011c7f88b0e0ef9',
   'proposal_key': 'sha256:19bfbdb65011c7f88b0e0ef95d491dc8462f70bc461fec8f47d45468af7a1340',
   'proposal_kind': 'predicate',
   'proposed_value': 'oc:unknown_predicate_demo',
   'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
   'target_pack': {'pack_id': 'onto_canon_psyop_seed__overlay',
    'pack_version': '0.1.0'},
   'reason': "default ontology mode 'mixed' resolved action 'propose' for predicate",
   'status': 'accepted',
   'application_status': 'recorded',
   'details': {'candidate_source_kind': 'notebook',
    'source': 'validate_assertion_payload'},
   'candidate_ids': ('cand_6a98976fcdce4db2997c74e5',),
   'created_at': '2026-03-17T21:27:55.060120Z',
   'review': {'decision_id': 'dec_2a6dd71769814871bf077d19',
    'proposal_id': 'prop_19bfbdb65011c7f88b0e0ef9',
    'decision': 'accepted',
    'actor_id': 'notebook:reviewer',
    'note_text': 'Reasonable local predicate for later on

In [8]:
try:
    service.review_proposal(
        proposal_id=accepted.proposal_id,
        decision="accepted",
        actor_id="notebook:other-reviewer",
    )
except Exception as exc:
    type(exc).__name__, str(exc)